In [1]:
from data_utils import *
import numpy as np
from tqdm import tqdm
import torch
import random
import pickle
import numpy as np
from collections import defaultdict
import scipy.sparse as sp

import os
import random
import pandas as pd
import json
import pickle
import gzip
from tqdm import tqdm
from utils import getBasicScores,getFairnessScores,area_curve_metric,seedSet

/home/sairamv/anaconda3/lib/python3.8/site-packages/setuptools/distutils_patch.py:25: UserWarning: Distutils was imported before Setuptools. This usage is discouraged and may exhibit undesirable behaviors or errors. Please use Setuptools' objects directly or at least import Setuptools first.
  warnings.warn(


In [2]:
DATASET = 'beauty'

In [3]:
DATA_PATH = '../data/'
POS_PATH = 'top-preds/stage-1-POS-only-EXPLS/'
NEG_PATH = 'top-preds/stage-1-NEG-only-EXPLS'
ZERO_PATH = 'top-preds/stage-1-ZERO-EXPLS/'

In [4]:
BATCH_SIZE=100

In [5]:
seed = 999
seedSet(seed)

# Explanation Loading

In [6]:
pos_expls = load_pickle(os.path.join(POS_PATH,f'DEEPFM-{DATASET}-preds.pkl'))

In [7]:
neg_expls = load_pickle(os.path.join(NEG_PATH,f'DEEPFM-{DATASET}-preds.pkl'))

In [8]:
zero_expls = load_pickle(os.path.join(ZERO_PATH,f'DEEPFM-{DATASET}-preds.pkl'))

In [9]:
def map_sorted_preds_to_ranked_items(preds_file):
    ui_scores = preds_file['ui_scores']
    preds = preds_file['preds']
    
    assert isinstance(ui_scores, dict)
    assert isinstance(preds, list) or isinstance(preds, np.ndarray)

    mapped_results = {}
    user_ids = list(ui_scores.keys())
    num_candidates = len(next(iter(ui_scores.values())))  # assume uniform candidate size

    for idx, user_id in tqdm(enumerate(user_ids)):
        user_pred_scores = np.array(preds[idx * num_candidates : (idx + 1) * num_candidates])
        item_rank_dict = ui_scores[user_id]  # {item_id: -rank}
        
        # Invert to get: {-1: item_id1, -2: item_id2, ..., -100: item_id100}
        rank_to_item = {rank: item for item, rank in item_rank_dict.items()}
        
        # Sort predictions in descending order
        sorted_scores = np.sort(user_pred_scores)[::-1]
        
        # Map highest score to -1, second to -2, ..., lowest to -num_candidates
        user_item_score_map = {}
        for i in range(1, num_candidates + 1):
            rank = -i
            item_id = rank_to_item[rank]
            score = sorted_scores[i - 1]
            user_item_score_map[item_id] = (score,rank)
        
        mapped_results[user_id] = user_item_score_map

    return mapped_results

In [10]:
def get_score(u,v,mapping):
    return round(mapping[u][v][0],5),mapping[u][v][1]

In [11]:
def get_top_k(mapping,user, k):
    lst = sorted(mapping[user].items(), key=lambda x:x[1][0],reverse=True)
    if k == -1:
        k = len(lst)
    return [x[0] for x in lst[:k]]

In [12]:
new_pos_expls = map_sorted_preds_to_ranked_items(pos_expls)
new_neg_expls = map_sorted_preds_to_ranked_items(neg_expls)
new_zero_expls = map_sorted_preds_to_ranked_items(zero_expls)

22363it [00:00, 26763.23it/s]
22363it [00:00, 27242.57it/s]
22363it [00:00, 27150.49it/s]


In [13]:
targetItems = readTargetItem(os.path.join(DATA_PATH,DATASET,"targetItems.txt"))
datamaps = load_json(os.path.join(DATA_PATH,DATASET,"datamaps.json"))
targetItems = [int(datamaps["item2id"][x]) for x in targetItems]
targetItems[:5]

[271, 3983, 1979, 7928, 3649]

# Top 1% Popular Item Selection

In [14]:
def targetItemSelect(data, popularThreshold=0.1):
    interact = data.matrix()
    userNum = interact.shape[0]
    itemNum = interact.shape[1]
    print("Data shape: ",interact.shape)
    targetNum = int(popularThreshold * itemNum)
    
    def getPopularItemId(n):
        """
        Get the id of the top n popular items based on the number of ratings
        :return: id list
        """
        return np.argsort(interact[:, :].sum(0))[0, -n:].tolist()[0]

    
    targetItem = random.sample(set(getPopularItemId(int(popularThreshold * itemNum))),targetNum)

    targetItem = [int(i + 1) for i in targetItem] # [data.id2item[str(i + 1)] for i in targetItem]
    
    print("# target items: ",len(targetItem))
    return targetItem

class DataCreator():
    def __init__(self):
        
        rev_data = load_pickle(os.path.join(DATA_PATH,DATASET,"review_splits.pkl"))
        print("Rev data keys: ",rev_data.keys())
        
        datamaps = load_json(os.path.join(DATA_PATH,DATASET,"datamaps.json"))
        print("Datamaps keys: ",datamaps.keys())
        
        self.training_data =  rev_data['train']
        self.val_data = rev_data['val']
        self.test_data = rev_data['test']
        
        self.user = datamaps['user2id']
        self.item = datamaps['item2id']
        self.id2user = datamaps['id2user']
        self.id2item = datamaps['id2item']
                
        self.__get_interact_data() # ADDED for convenience; changes the training data variable
                
        self.user_num = len(self.id2user)
        self.item_num = len(self.id2item)
        
        print("User Count:" ,self.user_num)
        print("Item Count:",self.item_num)
                
        
    def __get_interact_data(self):
        train_inter = []
        for pair in self.training_data:
            user,item,rating = pair['reviewerID'],pair['asin'],pair['overall']
            train_inter.append((user,item,rating))
        
        val_inter = []
        for pair in self.val_data:
            user,item,rating = pair['reviewerID'],pair['asin'],pair['overall']
            val_inter.append((user,item,rating))
        
        test_inter = []
        for pair in self.test_data:
            user,item,rating = pair['reviewerID'],pair['asin'],pair['overall']
            test_inter.append((user,item,rating))
            
        self.training_data = train_inter
        self.val_data = val_inter
        self.test_data = test_inter


    def __create_sparse_interaction_matrix(self):
        """
        return a sparse adjacency matrix with the shape (user number, item number)
        """
        row, col, entries = [], [], []
        for pair in self.training_data:
            row += [int(self.user[pair[0]]) - 1] # Correction July 22: apply correction (minus 1) for the index!
            col += [int(self.item[pair[1]]) - 1] # Correction July 22: apply correction (minus 1) for the index!
            entries += [1.0]
        
        interaction_mat = sp.csr_matrix((entries, (row, col)), shape=(self.user_num,self.item_num),dtype=np.float32)
        return interaction_mat


    def matrix(self):

        m = self.__create_sparse_interaction_matrix()
        return m

In [15]:
THRESH = 0.01
popItems = targetItemSelect(DataCreator(),popularThreshold=THRESH) # top 1%most popular items
popItems[:5]

Rev data keys:  dict_keys(['train', 'val', 'test', 'train_indices', 'val_indices', 'test_indices'])
Datamaps keys:  dict_keys(['user2id', 'item2id', 'id2user', 'id2item', 'attribute2id', 'id2attribute', 'attributeid2num'])
User Count: 22363
Item Count: 12101
Data shape:  (22363, 12101)
# target items:  121


[879, 6464, 2080, 2010, 3023]

In [16]:
sum([int(item in targetItems) for item in popItems])

121

# Re-ranking using the algorithm

In [17]:
def rerank_with_pop_demote(user, new_neg_expls, new_zero_expls, popItems):
    """
    Re-rank using zero explanation scores for all items,
    except for popular items where we use the negative explanation scores.

    Parameters:
    - user: the current user to check
    - new_neg_expls: dict {user: {item: (score, -rank)}}
    - new_zero_expls: dict {user: {item: (score, -rank)}}
    - popItems: set of item IDs

    Returns:
    - result: dict {user: {item: score}}
    """
    reranked_scores = {}

    for item in new_zero_expls[user]:
        if item in popItems:
            score,_ = get_score(user, item, new_neg_expls)
            reranked_scores[item] = score
        else:
            score, _ = get_score(user, item, new_zero_expls)
            reranked_scores[item] = score

    return reranked_scores

In [18]:
all_info = []
golds,preds = [],[]
pop_golds = []

ui_scores = dict()
gt = dict()
pop_gt = dict()
top = [1,2,3,5,10,20]

users = list(new_pos_expls.keys())
for stepv, user in tqdm(enumerate(users)):
    user = int(user)
    gold_item = int(zero_expls['gt'][user][0])
    scores_dict = rerank_with_pop_demote(user, new_neg_expls, new_zero_expls, popItems)
    rerank_lst = sorted(scores_dict.items(),key = lambda x:x[1], reverse=True)
    gt[user] = [gold_item]
    pop_gt[user] = list(popItems) # we only check pop item relevance!
    pred_dict = {}

    for j in range(len(rerank_lst)):

        item, score = rerank_lst[j]
        pred_dict[item] = -(j + 1)
        label = int(gold_item == item)
        pop_label = int(item in popItems)
        golds.append(label)
        pop_golds.append(pop_label)
        preds.append(score)
    
    ui_scores[user] = pred_dict

print("# golds: ",len(golds))
print("# pop golds: ",len(pop_golds))
print("# preds: ",len(preds))
print(f"# popular items overall across all users: {sum(pop_golds)}")
print("Original Recommendation Performance")
_, Recommendresults = getBasicScores(ui_scores, gt, top)
print("\nOriginal AUC: ",area_curve_metric(golds,preds)) 
print("Original Fairness Performance")
FairResults = getFairnessScores(ui_scores, targetItems, top, len(datamaps['item2id']))

22363it [00:11, 1947.65it/s]


# golds:  2236300
# pop golds:  2236300
# preds:  2236300
# popular items overall across all users: 23872
Original Recommendation Performance

NDCG@1	Rec@1	Hits@1	Prec@1	MAP@1	MRR@1
0.1319	0.1319	0.1319	0.1319	0.1319	0.1319

NDCG@2	Rec@2	Hits@2	Prec@2	MAP@2	MRR@2
0.1340	0.1353	0.1353	0.0676	0.1336	0.1336

NDCG@3	Rec@3	Hits@3	Prec@3	MAP@3	MRR@3
0.1363	0.1399	0.1399	0.0466	0.1351	0.1351

NDCG@5	Rec@5	Hits@5	Prec@5	MAP@5	MRR@5
0.1411	0.1517	0.1517	0.0303	0.1378	0.1378

NDCG@10	Rec@10	Hits@10	Prec@10	MAP@10	MRR@10
0.1542	0.1928	0.1928	0.0193	0.1430	0.1430

NDCG@20	Rec@20	Hits@20	Prec@20	MAP@20	MRR@20
0.1788	0.2917	0.2917	0.0146	0.1496	0.1496

Original AUC:  0.5546368880422567
Original Fairness Performance

PR@1	LTR@1	KLD@1	Gini@1	SDI@1	UHC@1
0.3919	0.6081	0.2968	0.4796	0.4766	0.3919

PR@2	LTR@2	KLD@2	Gini@2	SDI@2	UHC@2
0.3686	0.6314	0.2571	0.5289	0.4655	0.5770

PR@3	LTR@3	KLD@3	Gini@3	SDI@3	UHC@3
0.3562	0.6438	0.2368	0.5474	0.4586	0.6911

PR@5	LTR@5	KLD@5	Gini@5	SDI@5	UHC@5
0.3376	0.6624	0

In [19]:
print("Negative Explanation Ablation: \nRecommendation Performance wrt Popular Items")
_, Recommendresults = getBasicScores(ui_scores, pop_gt, top)
print("\nPopular Item AUC: ",area_curve_metric(pop_golds,preds))

Negative Explanation Ablation: 
Recommendation Performance wrt Popular Items

NDCG@1	Rec@1	Hits@1	Prec@1	MAP@1	MRR@1
0.0014	0.0000	0.0014	0.0014	0.0014	0.0014

NDCG@2	Rec@2	Hits@2	Prec@2	MAP@2	MRR@2
0.0020	0.0000	0.0023	0.0011	0.0019	0.0019

NDCG@3	Rec@3	Hits@3	Prec@3	MAP@3	MRR@3
0.0022	0.0000	0.0027	0.0009	0.0020	0.0020

NDCG@5	Rec@5	Hits@5	Prec@5	MAP@5	MRR@5
0.0032	0.0000	0.0052	0.0010	0.0026	0.0026

NDCG@10	Rec@10	Hits@10	Prec@10	MAP@10	MRR@10
0.0055	0.0001	0.0125	0.0013	0.0035	0.0035

NDCG@20	Rec@20	Hits@20	Prec@20	MAP@20	MRR@20
0.0134	0.0004	0.0445	0.0024	0.0055	0.0055

Popular Item AUC:  0.4308405410083238


In [20]:
print("Zero Score for all: \nRecommendation Performance wrt Popular Items")
_, Recommendresults = getBasicScores(zero_expls['ui_scores'], pop_gt, top)
print("\nPopular Item AUC: ",area_curve_metric(pop_golds,zero_expls['preds'])) 

Zero Score for all: 
Recommendation Performance wrt Popular Items

NDCG@1	Rec@1	Hits@1	Prec@1	MAP@1	MRR@1
0.0921	0.0008	0.0921	0.0921	0.0921	0.0921

NDCG@2	Rec@2	Hits@2	Prec@2	MAP@2	MRR@2
0.1262	0.0013	0.1462	0.0765	0.1191	0.1191

NDCG@3	Rec@3	Hits@3	Prec@3	MAP@3	MRR@3
0.1497	0.0017	0.1935	0.0702	0.1343	0.1349

NDCG@5	Rec@5	Hits@5	Prec@5	MAP@5	MRR@5
0.1792	0.0026	0.2675	0.0622	0.1486	0.1516

NDCG@10	Rec@10	Hits@10	Prec@10	MAP@10	MRR@10
0.2166	0.0042	0.3897	0.0505	0.1584	0.1679

NDCG@20	Rec@20	Hits@20	Prec@20	MAP@20	MRR@20
0.2426	0.0059	0.5018	0.0358	0.1580	0.1758

Popular Item AUC:  0.5035226115290653


In [21]:
def compute_avg_rank_score(ablation_score_map,popItems,K):
    print(f"K:{K}")
    pop_zero_score = []
    pop_neg_score = []
    pop_zero_rank = []
    pop_neg_rank = []
    pop_diff_score = []
    pop_diff_rank = []
    
    for user in ablation_score_map:
        item_maps = ablation_score_map[user]
        
        item_list = sorted(item_maps.items(),key = lambda x:x[1][0],reverse=True)
        item_list = [x[0] for x in item_list][:K]
        for item in item_list:
            if item in popItems:
                neg_score, neg_rank = item_maps[item]
                zero_score, zero_rank = get_score(user,item, new_zero_expls)
                zero_rank = int(-zero_rank)
                neg_rank = int(-neg_rank)
                pop_zero_score.append(zero_score)
                pop_neg_score.append(neg_score)
                pop_zero_rank.append(zero_rank)
                pop_neg_rank.append(neg_rank)
                
    print("\tlen(pop_neg_score): ",len(pop_neg_score))
    print("\tlen(pop_zero_score): ",len(pop_zero_score))
    print('\n')
    print(f"\tE(pop neg score): {np.mean(pop_neg_score):.6f}")
    print(f"\tE(pop zero score): {np.mean(pop_zero_score):.6f}")
    print(f"\tDiff between E(pop zero score) - E(pop neg score): {np.mean(pop_zero_score) - np.mean(pop_neg_score):.6f}")
    print('\n')
    print(f"\tE(pop neg rank): {np.mean(pop_neg_rank):.6f}")
    print(f"\tE(pop zero rank): {np.mean(pop_zero_rank):.6f}")
    print(f"\tDiff between E(pop_neg_rank) - E(pop_zero_rank): {np.mean(pop_neg_rank) - np.mean(pop_zero_rank):.6f}")

#     DEN = len(ablation_score_map) * K
#     print("\n WITH DEN: ",DEN)
#     print(f"\tE(pop neg score): {(sum(pop_neg_score)/DEN):.6f}")
#     print(f"\tE(pop zero score): {(sum(pop_zero_score)/DEN):.6f}")
#     print(f"\tDiff between E(pop zero score) - E(pop neg score): {(sum(pop_zero_score)/DEN) - (sum(pop_neg_score)/DEN):.6f}")
#     print('\n')
#     print(f"\tE(pop neg rank): {(sum(pop_neg_rank)/DEN):.6f}")
#     print(f"\tE(pop zero rank): {(sum(pop_zero_rank)/DEN):.6f}")
#     print(f"\tDiff between E(pop_neg_rank) - E(pop_zero_rank): {(sum(pop_neg_rank)/DEN) - (sum(pop_zero_rank)/DEN):.6f}")

    print('='*50)
    return

In [22]:
ablation_score_map = map_sorted_preds_to_ranked_items({'ui_scores':ui_scores,'gt':gt, 'golds':golds, 'preds':preds})

22363it [00:00, 26901.55it/s]


In [23]:
for K in [1,2,3,5,10,20]:
    compute_avg_rank_score(ablation_score_map,popItems,K)

K:1
	len(pop_neg_score):  32
	len(pop_zero_score):  32


	E(pop neg score): 1.000000
	E(pop zero score): 1.000000
	Diff between E(pop zero score) - E(pop neg score): 0.000000


	E(pop neg rank): 1.000000
	E(pop zero rank): 1.000000
	Diff between E(pop_neg_rank) - E(pop_zero_rank): 0.000000
K:2
	len(pop_neg_score):  51
	len(pop_zero_score):  51


	E(pop neg score): 0.999999
	E(pop zero score): 1.000000
	Diff between E(pop zero score) - E(pop neg score): 0.000001


	E(pop neg rank): 1.372549
	E(pop zero rank): 1.372549
	Diff between E(pop_neg_rank) - E(pop_zero_rank): 0.000000
K:3
	len(pop_neg_score):  61
	len(pop_zero_score):  61


	E(pop neg score): 0.999944
	E(pop zero score): 1.000000
	Diff between E(pop zero score) - E(pop neg score): 0.000056


	E(pop neg rank): 1.639344
	E(pop zero rank): 1.622951
	Diff between E(pop_neg_rank) - E(pop_zero_rank): 0.016393
K:5
	len(pop_neg_score):  116
	len(pop_zero_score):  116


	E(pop neg score): 0.990680
	E(pop zero score): 0.999998
	Diff betwe

In [24]:
for K in [100]:
    compute_avg_rank_score(ablation_score_map,popItems,K)

K:100
	len(pop_neg_score):  23872
	len(pop_zero_score):  23872


	E(pop neg score): 0.285152
	E(pop zero score): 0.829792
	Diff between E(pop zero score) - E(pop neg score): 0.544640


	E(pop neg rank): 57.453376
	E(pop zero rank): 20.337885
	Diff between E(pop_neg_rank) - E(pop_zero_rank): 37.115491


In [25]:
OUTPUT_DIR = os.path.join("top-preds","NEG-ABL")
os.makedirs(OUTPUT_DIR,exist_ok=True)

In [26]:
save_path = os.path.join(OUTPUT_DIR,f"DEEPFM-{DATASET}-preds-top-1%.pkl")
save_pickle({'ui_scores':ui_scores,
             'gt':gt, 'golds':golds, 
             'preds':preds, 'pop_golds':pop_golds,
             'pop_gt':pop_gt
            },
            save_path)